In [1]:
import json
import os
import chromadb
from typing import Annotated, TYPE_CHECKING

from IPython.display import display, HTML

from openai import AsyncOpenAI

from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.contents import FunctionCallContent,FunctionResultContent, StreamingTextContent, ChatMessageContent
from semantic_kernel.contents.utils.author_role import AuthorRole
from semantic_kernel.functions import kernel_function

from semantic_kernel.connectors.ai.open_ai import OpenAIChatPromptExecutionSettings
from semantic_kernel.contents.chat_history import ChatHistory
from semantic_kernel.contents import AuthorRole

if TYPE_CHECKING:
    from chromadb.api.models.Collection import Collection
# Initialize the asynchronous OpenAI client
from dotenv import load_dotenv

from typing import Annotated, Dict, Any, List
from semantic_kernel.functions import kernel_function

In [2]:
import json
import os

# ------------------------------
# 动态加载 subcommand 信息
# ------------------------------
def load_subcommands(info_path="../json/RNAFM_subcommand_info.json", param_path="../json/RNAFM_subcommand_parameter.json"):
    with open(info_path, "r") as f:
        subcommand_info = json.load(f)
    with open(param_path, "r") as f:
        subcommand_parameter = json.load(f)
    return subcommand_info, subcommand_parameter

# 使用示例
SUBCOMMANDINFO, subcommand_parameter = load_subcommands()

In [3]:
# ------------------------------
# 4️⃣ 加载 ChromaDB
# ------------------------------
chroma_client = chromadb.PersistentClient(path="../db/RNAFM_db")
collection = chroma_client.get_or_create_collection(
    name="RNAFM_documents",
    metadata={"description": "RNAFM_import_documents"}
)
# 插入示例文档
documents = [
    "RNAFM has two types of functions: those that require training (classify, predict_expression) and those that use pre-trained models directly (embed, predict_ss, cluster).",
    "For training tasks (classify, predict_expression), GPU is strongly recommended for faster processing.",
    "The predict_ss command supports intelligent output: if you specify a directory path, it automatically generates all formats (.png and .ct) using sequence IDs as filenames.",
    "Model type selection: 'ss' for secondary structure focus, 'rna' for general RNA (default), 'mrna' for mRNA-specific tasks.",
    "For cluster and classify commands, input must be a folder containing RF*.fasta files, where each file represents one RNA family.",
    "For predict_expression, input must be a CSV file with 'Sequence' and 'Value' columns, optionally with 'splits' column for train/val/test splits.",
]
collection.add(
    documents=documents,
    ids=[f"doc_{i}" for i in range(len(documents))],
    metadatas=[{"source": "RNAFM_usage"} for _ in documents],
)

In [4]:
import json
from typing import Dict, Any

class CodeGeneratorPlugin:
    def __init__(self, collection: "Collection", subcommand_parameter_dict: Dict[str, Any] = None):
        self.subcommand_parameter = subcommand_parameter_dict
        self.collection = collection

    @kernel_function(
        description="Get the subcommand parameter",
        name="get_subcommand_parameter"
    )
    def get_subcommand_parameter(self, subcommand_name: str) -> Dict[str, Any]:
        return self.subcommand_parameter[subcommand_name]


    @kernel_function(
        description="Get the document context",
        name="get_document_context"
    )
    def get_document_context(self, user_query: str) -> str:
        return self.collection.query(query_texts=[user_query], n_results=10)


    @kernel_function(
        description="Generate a prompt for the LLM given tool parameters JSON and the user's task request.",
        name="generate_tool_prompt"
    )
    def generate_tool_prompt(self,tool_definition: Dict[str, Any], user_task: str) -> str:
        """
        Generate a prompt for the LLM given tool parameters JSON and the user's task request.
        English only.
        """
        formatted_params = json.dumps(tool_definition, indent=2, ensure_ascii=False)
        prompt = (
            "You are an expert systems engineer skilled in integrating command-line tools. "
            "You will be given a parameter definition and a user task. "
            "Your goal is to generate accurate codes that fulfills the user's request.\n\n"
            f"User Task:\n{user_task}\n\n"
            "Parameter Definition:\n"
            f"{formatted_params}\n\n"
            "Instructions:\n"
            "1. Use relevant optional parameters if they match the user's intent.\n"
            "2. Only use the parameters that are shown in the parameter definition. Never use parameters that are not in the parameter definition."
            "3. Do not use space as a parameter value. Use a name instead."
        )
        return prompt
    


In [5]:
# ------------------------------
# 1️⃣ 初始化 OpenAI 客户端
# ------------------------------
load_dotenv()
client = AsyncOpenAI(
    api_key=os.environ["GITHUB_TOKEN"],
    base_url="https://models.inference.ai.azure.com/"
)

chat_completion_service = OpenAIChatCompletion(
    ai_model_id="gpt-4o",
    async_client=client,
)

In [6]:
from pydantic import BaseModel, ValidationError, Field

class SubTask(BaseModel):
    assigned_subcommand: str = Field(
        description="The specific subcommand assigned to handle this subtask")
    task_details: str = Field(
        description="Detailed description of what needs to be done for this subtask")


class PreparePlan(BaseModel):
    main_task: str = Field(
        description="The overall request from the user")
    subtasks: List[SubTask] = Field(
        description="List of subtasks broken down from the main task, each assigned to a specialized subcommand")

In [7]:
from semantic_kernel.functions import KernelArguments
AGENT_NAME = "Prepare_Agent"

AGENT_INSTRUCTIONS = """You are an planner agent.
    Your job is to decide which subcommand to run based on the user's request.
    Below are the available agents specialised in different tasks:
"""

for name, info in SUBCOMMANDINFO.items():
    AGENT_INSTRUCTIONS += ("\n - "+name+": "+info.get("description")+"\t"+ "Input: "+info.get("input"))


# Create the prompt execution settings and configure the Pydantic model response format
settings = OpenAIChatPromptExecutionSettings(response_format=PreparePlan)

pre_agent = ChatCompletionAgent(
    service=chat_completion_service,
    description="You are an planner agent.",
    name=AGENT_NAME,
    instructions=AGENT_INSTRUCTIONS,
    arguments=KernelArguments(settings) 
)

In [8]:
from semantic_kernel.functions import KernelArguments
AGENT_NAME = "CodeGeneratorAgent"

AGENT_INSTRUCTIONS = """
    ### Role
    You are the RNAFM Code Generator Agent, an expert in RNA sequence analysis using pre-trained RNA-FM models. You generate precise commands for RNA embedding, secondary structure prediction, clustering, classification, and expression prediction.

    ### Workflow

    #### 1. Smart Information Audit (The Delta Check)
    Analyze input to identify **MISSING** variables. Acknowledge what is known; only ask for the unknown:
    * **Task Type**: Which subcommand? (embed, predict_ss, cluster, classify, predict_expression)

    #### 2. Code Generation (Once all variables are known)
    
    **Key Rules:**
    - **Command Format**: Use `ReRNAFM <subcommand> [options]`
    - **Model Type Selection**:
        - `ss`: For secondary structure-focused tasks
        - `rna`: For general RNA analysis (default)
        - `mrna`: For mRNA-specific tasks (recommended for predict_expression)
    - **Intelligent Output**: For `predict_ss`, if output is a directory path, it automatically generates all formats (.png, .ct) using sequence IDs as filenames
    - **Device**: Default to 'cuda' if available, otherwise 'cpu'. For training tasks (classify, predict_expression), strongly recommend GPU.
    - **Syntax**: Wrap code in <code>...</code>. Use `[INPUT_PATH]`, `[OUTPUT_PATH]`, `[SEQUENCE]` as placeholders.

    #### 3. Training vs Non-Training Tasks
    - **No Training Required** (use pre-trained models directly):
        - `embed`: Generates embeddings immediately
        - `predict_ss`: Predicts secondary structure immediately
        - `cluster`: Performs clustering visualization immediately
    - **Training Required** (needs training step):
        - `classify`: Trains RNA family classifier (requires *.fasta folder)
        - `predict_expression`: Trains expression prediction model (requires CSV with Sequence, Value)

    #### 4. Proactive Expert Suggestions (Post-Command)
    After providing the code, offer these optional value-adds:
    - **For embed**: "The embeddings can be used for downstream tasks like clustering or classification. Would you like me to generate a clustering command next?"
    - **For predict_ss**: "The structure predictions include probability matrices, visualization plots, and structure files. If you specified a directory, all formats are automatically generated."
    - **For cluster**: "The t-SNE visualization shows RNA family distributions. You can adjust `--n_components` for 3D visualization or `--random_state` for reproducibility."
    - **For classify/predict_expression**: "Training may take time depending on data size. GPU is strongly recommended. The model checkpoint will be saved for future predictions."

    ---

    ### Interaction Protocol Examples

    **User:** "I want to predict secondary structure for my RNA sequences in sequences.fasta"
    **Agent:** "Got it. Preparing the command for RNA secondary structure prediction.
    
    <code>ReRNAFM predict_ss --sequences_file sequences.fasta --output ./results/</code>
    
    This will automatically generate .png visualization and .ct structure files for each sequence in the ./results/ directory, using sequence IDs as filenames.
    
    **Expert Tips:**
    - **Output Format**: If you specify a directory (./results/), all formats (.png and .ct) are automatically generated. If you specify a file path, only that format is saved.
    - **Device**: If you have GPU available, add `--device cuda` for faster processing.
    - **Next Step:** Would you like to generate embeddings for these sequences for further analysis?"

    **User:** "I want to embed RNA sequences"
    **Agent:** "I see you want to generate RNA sequence embeddings. Do you have:
    1. Sequences to provide directly (--sequences) or a sequence file (--sequences_file)?
    2. A preference for model type? Use 'rna' for general RNA (default), 'ss' for secondary structure focus, or 'mrna' for mRNA-specific sequences.
    3. An output path to save the embeddings (.npy file)?"

    **User:** "I want to train a classifier for RNA families"
    **Agent:** "I see you want to train an RNA family classifier. This requires:
    1. GPU is strongly recommended for training. Do you have CUDA available?
    2. Where should I save the trained model checkpoint?"
"""

code_agent = ChatCompletionAgent(
    service=chat_completion_service,
    description="You are an code generator agent.",
    name=AGENT_NAME,
    instructions=AGENT_INSTRUCTIONS,
    plugins=[CodeGeneratorPlugin(collection, subcommand_parameter)],
)

In [9]:
from semantic_kernel.agents import SequentialOrchestration, GroupChatOrchestration, RoundRobinGroupChatManager

from semantic_kernel.contents import ChatMessageContent

def agent_response_callback(message: ChatMessageContent) -> None:
    print(f"# {message.name}\n{message.content}")


In [10]:
from semantic_kernel.contents import ChatHistorySummarizationReducer

# Configure reduction parameters
REDUCER_TARGET_COUNT = 1  # Target number of messages to keep after reduction
REDUCER_THRESHOLD =  4 # Trigger reduction when message count exceeds this

history_reducer = ChatHistorySummarizationReducer(
    service=chat_completion_service,
    target_count=REDUCER_TARGET_COUNT,
    threshold_count=REDUCER_THRESHOLD,
)

chat = SequentialOrchestration(
    members=[pre_agent, code_agent],
    agent_response_callback=agent_response_callback,
)


In [ ]:
from semantic_kernel.agents.runtime import InProcessRuntime
runtime = InProcessRuntime()
runtime.start()

# Prepare_Agent
{"main_task":"Classify RNA sequences into families.","subtasks":[{"assigned_subcommand":"classify","task_details":"Use the provided RNA family FASTA files to train a classifier for categorizing RNA sequences into different RNA families."}]}
# CodeGeneratorAgent
It seems you want to classify RNA sequences into different families using RNA-FM. I need to confirm some additional details to generate the precise command:

1. **Input Data**: Please provide a folder containing the RNA family training *.fasta files (`--data_dir`).
2. **Device**: Do you have access to a GPU for training, or should I default to CPU?
3. **Output Checkpoint**: Where should the trained model checkpoint be saved (`--save_model_path`)?

Let me know these details, and I'll generate the finalized command for you!
# Prepare_Agent
{"main_task":"Classify RNA sequences into families.","subtasks":[{"assigned_subcommand":"classify","task_details":"Process the input FASTA file 'sequences.fasta' for classifying 

In [12]:
user_inputs = [
    "I want to classify my RNA sequences",
    "My sequences are in sequences.fasta file, output to ./results/ directory",
    
]

async def main():
    thread = ChatHistoryAgentThread(chat_history=history_reducer)
    for user_input in user_inputs:
        history_reducer.add_user_message(user_input)
        orchestration_result = await chat.invoke(
            task=history_reducer.messages,
            runtime=runtime,
        )
        value = await orchestration_result.get(timeout=100)
        print(f"***** Final Result *****\n{value}")
        history_reducer.add_assistant_message(value.content)

        if len(thread) > 4:
            await thread.reduce()
    await runtime.stop_when_idle()

await main()

***** Final Result *****
It seems you want to classify RNA sequences into different families using RNA-FM. I need to confirm some additional details to generate the precise command:

1. **Input Data**: Please provide a folder containing the RNA family training *.fasta files (`--data_dir`).
2. **Device**: Do you have access to a GPU for training, or should I default to CPU?
3. **Output Checkpoint**: Where should the trained model checkpoint be saved (`--save_model_path`)?

Let me know these details, and I'll generate the finalized command for you!
***** Final Result *****
To train an RNA family classifier, I need the following details:

1. The **path to the folder containing RNA family FASTA files** (`--fasta_folder`). Each file should represent an RNA family and follow the format `RF*.fasta` (e.g., RF00001.fasta).
2. The **path to save the trained model checkpoint** (`--checkpoint_path`). If not specified, the default is `rna_family_classifier.pt`.
3. The **model type** for embeddings 